In [48]:
import pandas as pd
import pycountry

In [49]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
            "prepare_wacc",
    )

### File paths

In [50]:
wacc_fn = snakemake.input.wacc
bus_locations_fn = snakemake.input.bus_locations

# Output

wacc_clustered_fn = snakemake.output.wacc

# config
config = snakemake.config

### Get WACC data

In [51]:
# Read the CSV data
# wacc = pd.read_csv(wacc_fn) #, index_col='Region'
wacc = pd.read_csv(wacc_fn, comment='#', index_col="iso_alpha")


### Cluster per region


Map each country in the production data to its region using the region definitions from the config file.


In [52]:
# Load region definitions from config file
regions = config['regions']

In [53]:
country_name_corrections = {
    "Democratic Republic of the Congo": "Congo, The Democratic Republic of the",
    "Republic of the Congo": "Republic of the Congo",
    "Kosovo": "Republic of Kosovo",  # pycountry not supported
    "Russia": "Russian Federation",
    "Turkey": "Türkiye",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Tanzania": "United Republic of Tanzania",
    "Bolivia": "Plurinational State of Bolivia",
    "Vietnam": "Viet Nam",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Taiwan": "Taiwan, Province of China",
    "Laos": "Lao People's Democratic Republic",
    "Brunei": "Brunei Darussalam",
    "Equatorial French Guiana": "French Guiana",
    "Syria": "Syrian Arab Republic",   
    "Palestine": "Palestine, State of",
    "Moldova": "Republic of Moldova",
    "North Korea": "Korea, Democratic People's Republic of",
}

In [54]:
# Apply a country name correction to the regions dict
for region, countries in regions.items():
    for i, country in enumerate(countries):
        if country in country_name_corrections:
            countries[i] = country_name_corrections[country]

In [55]:
# Build a mapping from country name to ISO_A2 code
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_3
    # Add common names
    if hasattr(country, 'official_name'):
        country_name_to_iso[country.official_name] = country.alpha_3

# Build a mapping from ISO_A2 code to region
iso_to_region = {}
for region, countries in regions.items():
    for country in countries:
        # Some country names may have extra text (e.g., "Togo + Algeria"), handle them simply
        for part in country.split('+'):
            name = part.strip()
            code = country_name_to_iso.get(name)
            if code:
                iso_to_region[code] = region

In [56]:
# Map each row in production to its region
wacc['region'] = wacc.index.map(lambda iso: iso_to_region.get(iso, 'Other'))
# wacc.sort_values(by='SteelDemandMt', ascending=False)

In [57]:
# Filter for relevant columns
wacc = wacc[["wacc_real", "region"]]

In [58]:
wacc_by_region = wacc.groupby('region').mean(numeric_only=True)
wacc_by_region

,wacc_real
region,
Central_America,0.095276
East_Asia,0.089522
East_East_Asia,0.078482
Eurasia,0.103032
Europe,0.095197
Far_West_Europe,0.084226
Middle_East,0.118759
North_America,0.076840
North_West_Africa,0.116947


### Save clustered WACC

In [59]:
wacc_by_region.to_csv(wacc_clustered_fn)